# 4. Choosing a sampling strategy and a surrogate

bdopt has nine adaptive sampling strategies and fifteen surrogate models. This tutorial compares them on a
cheap analytic response (no EnergyPlus), so you can see how to run such a comparison on your own model: the
only thing that changes is the evaluator. The response has a narrow bump on a smooth trend and a mild
oscillation in the third parameter.

In [1]:
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 4)
pd.set_option("display.width", 120)

In [2]:
from bdopt import FunctionEvaluator, NullSelector, Parameter, Problem, RangeParameter, sampling

params = [Parameter(NullSelector(n), RangeParameter(0, 1), name=n) for n in ("x1", "x2", "x3")]
problem = Problem(params, ["f1", "f2"])

def simulator(row):
    bump = np.exp(-(((row["x1"] - 0.7) ** 2 + (row["x2"] - 0.3) ** 2) / 0.02))
    return [0.3 * row["x1"] + bump + 0.1 * np.sin(6 * row["x3"]), row["x2"] ** 2 + 0.5 * row["x3"] + 0.4 * bump]

evaluator = FunctionEvaluator(problem, simulator)

test = sampling.lhs(problem, 2000, seed=99)               # independent points to judge every model
test = test.join(evaluator.evaluate(test))

## Sampling strategies

`bdopt.adaptive.METHODS` lists them. Every method starts from the same 12 Latin-hypercube simulations plus 6
held-out ones and adds 20 more, one at a time; the surrogate is the same Gaussian process. The error is the RMSE
on the 2000 independent points divided by the standard deviation of each objective, averaged over objectives.

In [3]:
from bdopt.adaptive import METHODS, AdaptiveSampler
from bdopt.surrogate import GPSurrogate

def error(model):
    s = model.score(test)
    return float((s["rmse"] / test[problem.objective_names].std()).mean())

rows = []
for method in METHODS:
    errors = []
    for seed in range(2):
        sampler = AdaptiveSampler(problem, evaluator, method, GPSurrogate(n_restarts=2, max_extra_restarts=6), seed=seed)
        result = sampler.run(n_initial=12, n_test=6, max_iter=20, tol=0, verbose=False)
        errors.append(error(result.surrogate))
    rows.append((method, np.mean(errors), np.std(errors)))

plain = []
for seed in range(2):
    train = sampling.lhs(problem, 32, seed=seed)
    train = train.join(evaluator.evaluate(train))
    plain.append(error(GPSurrogate(n_restarts=2, max_extra_restarts=6).fit(train, problem)))
rows.append(("plain Latin hypercube (32)", np.mean(plain), np.std(plain)))
pd.DataFrame(rows, columns=["method", "mean error", "std over seeds"]).set_index("method").sort_values("mean error")

,mean error,std over seeds
method,,
flola_voronoi,0.2941,0.0123
random,0.2998,0.0164
plain Latin hypercube (32),0.3363,0.0267
cv_voronoi,0.3533,0.0128
lola_voronoi,0.3664,0.0066
voronoi,0.3750,0.0580
latin_hypercube,0.4281,0.0062
gp_variance,0.4364,0.0484
nipv,0.4630,0.0179


With two seeds the differences between neighbours are inside the spread: read the table as "which methods
are clearly worse", not as a ranking. `docs/benchmarks/platform_benchmark.py` does this on EnergyPlus models.

## Surrogate models

Same 40 Latin-hypercube simulations for every model, error on the independent points.

In [4]:
import time
from bdopt.surrogate import SURROGATES, make_surrogate

names = list(SURROGATES) + ["torch_mlp", "mc_dropout_mlp", "multitask_gp", "deep_kernel_gp"]
train = sampling.lhs(problem, 40, seed=0)
train = train.join(evaluator.evaluate(train))
rows = []
for name in names:
    t0 = time.time()
    model = make_surrogate(name).fit(train, problem)
    rows.append((name, error(model), model.has_uncertainty, time.time() - t0))
pd.DataFrame(rows, columns=["surrogate", "error", "uncertainty", "fit seconds"]).set_index("surrogate").sort_values("error")

,error,uncertainty,fit seconds
surrogate,,,
auto,0.2373,True,16.3578
gp,0.2373,True,0.5825
rbf,0.2905,False,0.0071
gradient_boosting,0.3104,False,0.2595
deep_kernel_gp,0.3284,True,0.5499
extra_trees,0.3406,True,0.1652
kernel_ridge,0.3620,False,0.2703
random_forest,0.4050,True,0.2211
mc_dropout_mlp,0.4538,True,0.4691


## Letting bdopt choose

{class}`~bdopt.surrogate.AutoSurrogate` cross-validates a list of candidates on the training data only and keeps
the best.

In [5]:
from bdopt.surrogate import AutoSurrogate

auto = AutoSurrogate(cv=5).fit(train, problem)
print("selected:", auto.selected_, "  error on the independent points:", round(error(auto), 4))
auto.cv_scores_

selected: GPSurrogate   error on the independent points: 0.2373


GPSurrogate             0.4976
KernelRidgeSurrogate    0.5148
ExtraTreesSurrogate     0.5375
MLPSurrogate            0.5474
PolynomialSurrogate     0.5617
SVRSurrogate            0.5927
RBFSurrogate            0.6384
dtype: float64

## Next

* How each strategy works: {doc}`../guide/adaptive-sampling`; each model: {doc}`../guide/surrogates`.
* On EnergyPlus models: {doc}`02_office_building`.